# Extract sample GT and append annotation

In [ ]:
import os
import pandas as pd
import polars as pl
import subprocess
from concurrent.futures import ThreadPoolExecutor
import openpyxl
from openpyxl import Workbook

import subprocess

n_workers = 6
threads = str(1)

# A single plink file or separate for chromosomes
inputPfile = 'input_filename'

os.chdir('/path/to/input/files')

In [ ]:
# Get if file with plink2 --write-samples command
samples = pd.read_csv(f'{inputPfile}.id', sep='\t', header=0)
samples

### Extract variants with mac 1 and 2 in each sample

In [ ]:
def run_plink(sample):
    subprocess.run([
        "plink2",
        "--pfile", inputPfile,
        "--indv", sample,
        "--mac", "1",
        "--max-mac", "2",
        "--chr", "1-22",
        "--threads", str(threads),
        "--make-just-pvar",
        "--export", "vcf",
        "--out", f"./per_sample/{sample}"
    ], check=True)

with ThreadPoolExecutor(max_workers=n_workers) as ex:
    ex.map(run_plink, samples["#IID"])

### Get avinput

In [ ]:
df = pd.read_csv(f'{inputPfile}.pvar', sep='\t', header=0)
df_avinput = df[['#CHROM', 'POS', 'POS', 'REF',	'ALT']]
df_avinput.to_csv(f'{inputPfile}.avinput', sep='\t', header=False, index=False)

### Run annovar

In [ ]:
! for i in {1..22} ; do table_annovar.pl {inputPfile}.avinput ./annovar_2025/humandb -buildver hg38 -out {inputPfile} -remove -protocol refgene,avsnp151,clinvar_20250721,allofus,dbnsfp47a,gnomad41_genome,gnomad41_exome -operation g,f,f,f,f,f,f -nastring . -thread 8 ; done

### Parse outputs

In [ ]:
anno_cols = ['Chr', 'Start', 'Ref', 'Alt',
             'avsnp151', 'Gene.refGene', 'Func.refGene', 
             'ExonicFunc.refGene', 'AAChange.refGene', 
             'CLNSIG', 
             'CADD_raw', 'CADD_phred',
             'SIFT_score', 'SIFT_pred', 
             'Polyphen2_HDIV_score', 'Polyphen2_HDIV_pred', 
             'Polyphen2_HVAR_score','Polyphen2_HVAR_pred',
             'REVEL_score', 'REVEL_rankscore',
             'PrimateAI_score', 'PrimateAI_pred',
             'AlphaMissense_score', 'AlphaMissense_pred',
             'GTEx_V8_eQTL_gene', 'GTEx_V8_eQTL_tissue',
             'gnomad41_genome_AF','gnomad41_exome_AF',
             'gvs_all_af'
            ]

In [ ]:
anno = pd.read_csv(f'{inputPfile}.hg38_multianno.txt', sep='\t',header=0,
                 usecols=anno_cols).rename(columns={'Chr': 'CHR', 
                                                    'Start': 'POS', 
                                                    'Ref': 'REF', 
                                                    'Alt': 'ALT'})
anno['SNP'] = anno['CHR'].astype(str) + ":" + anno['POS'].astype(str) + ":"  + anno['REF'].astype(str) + ":"  + anno['ALT'].astype(str)
print(anno.shape[0])
chrom = list(range(1, 23))
anno = anno[anno['CHR'].isin(chrom)]
anno = pl.from_pandas(anno)

### Merge vcf and anno per sample

In [ ]:
def get_anno(sample):
    df = pd.read_csv(f'./per_sample/{sample}.vcf', separator='\t', comment="#", header=None, 
                     names=['CHR', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT', 'GT'],
                     usecols=['CHR', 'POS', 'ID', 'REF', 'ALT', 'GT'])
    df['SNP'] = df['CHR'].astype(str) + ":" + df['POS'].astype(str) + ":" + df['REF'].astype(str) + ":"  + df['ALT'].astype(str)
    df['SAMPLE'] = sample
    merge = pd.merge(df, anno, how='left', on=['CHR', 'POS', 'REF', 'ALT'])


with ThreadPoolExecutor(max_workers=n_workers) as ex:
    ex.map(get_anno, samples["#IID"])

In [ ]:
def get_anno(sample):
    df = (
        pl.read_csv(
            f"./per_sample/{sample}.vcf",
            separator="\t",
            comment_prefix="#",
            has_header=False,
            new_columns=["CHR","POS","ID","REF","ALT","QUAL","FILTER","INFO","FORMAT","GT"]
        )
        .select(["CHR","POS","ID","REF","ALT","GT"])
        .with_columns([
            (pl.col("CHR").cast(pl.Utf8) + ":" +
             pl.col("POS").cast(pl.Utf8) + ":" +
             pl.col("REF") + ":" +
             pl.col("ALT")).alias("SNP"),
            pl.lit(sample).alias("SAMPLE")
        ])
    )

    out = df.join(anno, on=["CHR","POS","REF","ALT"], how="left")
    out.write_csv(f"./per_sample/{sample}_anno.csv", separator='\t')

with ThreadPoolExecutor(max_workers=n_workers) as ex:
    results = list(ex.map(get_anno, samples["#IID"]))

### Write excel outputs per sample with pathogenic variant subsets

In [ ]:
def write_sample_xlsx(df_sample, sample):
    s2 = df_sample.filter(
        pl.col("CLNSIG").str.contains("pathogenic", case=False) &
        ~pl.col("CLNSIG").str.contains("conflicting", case=False)
    )

    pred_or = (
        pl.col("SIFT_pred").is_in(["D","."]) |
        pl.col("Polyphen2_HVAR_pred").is_in(["D","."]) |
        pl.col("Polyphen2_HDIV_pred").is_in(["D","."]) |
        pl.col("PrimateAI_pred").is_in(["D","."]) |
        pl.col("AlphaMissense_pred").is_in(["P","A","."]) |
        (pl.col("CADD_phred").is_null() | (pl.col("CADD_phred") >= 12))
    )

    pred_and = (
        pl.col("SIFT_pred").is_in(["D","."]) &
        pl.col("Polyphen2_HVAR_pred").is_in(["D","."]) &
        pl.col("Polyphen2_HDIV_pred").is_in(["D","."]) &
        pl.col("PrimateAI_pred").is_in(["D","."]) &
        pl.col("AlphaMissense_pred").is_in(["P","A","."]) &
        (pl.col("CADD_phred").is_null() | (pl.col("CADD_phred") >= 12))
    )

    s3 = df_sample.filter(pred_or)
    s4 = df_sample.filter(pred_and)

    wb = Workbook()
    wb.remove(wb.active)

    for name, d in [
        ("all_anno", df_sample),
        ("pathogenic", s2),
        ("pred_OR", s3),
        ("pred_AND", s4),
    ]:
        ws = wb.create_sheet(name)
        ws.append(d.columns)
        for row in d.to_numpy():
            ws.append(row.tolist())

    wb.save(f"./per_sample/{sample}.xlsx")


for sample in final_df.select("SAMPLE").unique().to_series():
    write_sample_xlsx(final_df.filter(pl.col("SAMPLE") == sample), sample)